# JEI Stock Radar — Colab 風險與資料驗證
讀取 GitHub main 的 `remote/system.json`，檢查日期、風險訊號與盤中行情狀態。**不會發送 LINE，也不會修改正式系統。**
執行前可在 Colab 選「執行階段 → 全部執行」。

In [ ]:
import json, urllib.request, datetime, csv
from zoneinfo import ZoneInfo
URL='https://raw.githubusercontent.com/alwaysjei1234-sys/JEI-Stock-Radar/main/remote/system.json'
with urllib.request.urlopen(URL,timeout=20) as response:
    feed=json.load(response)
assert isinstance(feed,dict), 'feed must be an object'
print('Data date:',feed.get('data_date'),'Updated:',feed.get('updated_at'))


In [ ]:
risk=feed.get('risk') or {}
market=feed.get('market') or {}
issues=[]
if not feed.get('data_date'): issues.append('缺少 data_date')
if risk.get('score') is None: issues.append('缺少 risk.score')
if not isinstance(risk.get('reasons'),list): issues.append('缺少 risk.reasons')
if market.get('live_valid') is False and market.get('live_count',0)>0: issues.append('盤中數量非零但 live_valid=false，需人工確認')
now=datetime.datetime.now(ZoneInfo('Asia/Taipei'))
try:
    age=(now.date()-datetime.date.fromisoformat(feed['data_date'])).days
    if age<0: issues.append('資料日期晚於台北今日')
    if age>4: issues.append(f'資料日期距今 {age} 天，可能過期（假日需人工確認）')
except (KeyError,ValueError,TypeError): issues.append('data_date 無法解析')
print('Risk:',risk.get('level'),risk.get('score'))
print('Market:',market.get('status'),'live_valid:',market.get('live_valid'))
print('Issues:',issues or '無基本結構異常')
if risk.get('level')=='red' or (isinstance(risk.get('score'),(int,float)) and risk['score']>=80):
    print('ALERT CANDIDATE: 高風險，應檢查正式通知管線；本筆記本不代表已推播')


In [ ]:
from pathlib import Path
out=Path('jei_colab_audit.csv')
with out.open('w',newline='',encoding='utf-8-sig') as f:
    writer=csv.writer(f)
    writer.writerow(['checked_at_taipei','data_date','risk_level','risk_score','market_status','live_valid','issues'])
    writer.writerow([now.isoformat(),feed.get('data_date'),risk.get('level'),risk.get('score'),market.get('status'),market.get('live_valid'),' | '.join(issues)])
print('Saved:',out)
